In [33]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, RocCurveDisplay, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

Bias refers to the error introduced by approximating a complex real-world problem with a simplified model, while variance refers to the model's sensitivity to fluctuations in the training data. A linear regression model has high bias and low variance; it makes strong assumptions about the data (linearity) but is stable across different datasets. If these strong assumptions are not correct, there will be places where it systematically overestimates or underestimates. In contrast, a decision tree model has low bias and high variance;it can capture complex patterns but is prone to overfitting, especially if deep and unpruned. This means that it can start to memorize the training data rather than capturing patterns that generalize.

In [2]:
kc_sales = pd.read_csv('data/kc_house_data.csv')

In [3]:
kc_sales

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,20140521T000000,360000.0,3,2.50,1530,1131,3.0,0,0,...,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,6600060120,20150223T000000,400000.0,4,2.50,2310,5813,2.0,0,0,...,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,1523300141,20140623T000000,402101.0,2,0.75,1020,1350,2.0,0,0,...,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,291310100,20150116T000000,400000.0,3,2.50,1600,2388,2.0,0,0,...,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


In [4]:
predictor_variables = ['sqft_living']
target = 'price'

X = kc_sales[predictor_variables]
y = kc_sales[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 321, test_size=0.3)

In [5]:
lr = LinearRegression().fit(X_train, y_train)

In [6]:
print(f'MSE on training data: {mean_squared_error(y_train, lr.predict(X_train))}')
print(f'MSE on test data: {mean_squared_error(y_test, lr.predict(X_test))}')

MSE on training data: 69332030732.38745
MSE on test data: 66079560485.167015


### 2. Repeat this but with a [DecisionTreeRegresor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html). Again check the mean squared error on the training data and the test data. How does what you see differ from the linear regression model?

In [7]:
dt = DecisionTreeRegressor().fit(X_train, y_train)

print(f'MSE on training data: {mean_squared_error(y_train, lr.predict(X_train))}')
print(f'MSE on test data: {mean_squared_error(y_test, lr.predict(X_test))}')

MSE on training data: 69332030732.38745
MSE on test data: 66079560485.167015


One way of avoiding overfitting is by restricting the flexibility of the model. We can do this with a decision tree by restricting the number of splits that it can perform.

### 3. Fit a DecisionTreeRegressor where you restrict the max_depth to 5. Again check the mean squared error on the training data and the test data. What do you notice now?

In [8]:
dt = DecisionTreeRegressor(max_depth=5).fit(X_train, y_train)

print(f'MSE on training data: {mean_squared_error(y_train, dt.predict(X_train))}')
print(f'MSE on test data: {mean_squared_error(y_test, dt.predict(X_test))}')

MSE on training data: 60841449026.33667
MSE on test data: 62926561451.64904


When working with machine learning models, we often have to balance bias and variance. This is called the [bias-variance tradeoff](https://en.wikipedia.org/wiki/Bias%E2%80%93variance_tradeoff). One method of this is through [regularization](https://www.ibm.com/think/topics/regularization), where we restrict the complexity of the model, adding some bias but reducing the variance, which can lead to a lower mean squared error on the test set.

Lasso and ridge regression do this by adding a penalty term based on the size of the coefficients. Smaller coefficients means that the model has less flexibility. The neat thing about these types of models is that they determine how to allocate the coefficients automatically as part of the model fitting process, so we can start with a large set of potential predictors and allow the model fitting to determine which ones to focus on.

For the next part of the exercise, we'll see how we can add complexity to our model but control the complexity through regularization.

### 4. So far, we've only been predicting based off of the square footage of living space. Fit a new linear regression model using all variables besides id, date, price, and zipcode. How well does this model perform on the test set compared to the model with just square footage of living space?

In [9]:
predictor_variables = [col for col in kc_sales.columns if col not in ['id', 'date', 'price', 'zipcode']]
target = 'price'

X = kc_sales[predictor_variables]
y = kc_sales[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 321, test_size=0.3)

In [10]:
X_train.columns

Index(['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long',
       'sqft_living15', 'sqft_lot15'],
      dtype='object')

In [11]:
lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f'MSE : {mean_squared_error(y_test, y_pred)}')
print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
print(f'R2 : {r2_score(y_test, y_pred)}')

MSE : 41757655724.75375
MSE : 125820.15889970955
R2 : 0.6844306976306158


### 5. Try fitting a lasso and ridge model. Becuase lasso and ridge have penalty terms based on the size of the coefficients, and the size of the coefficients depends on the scale of the variable, you'll want to scale the features first so that they are on comparable scales. Create a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) object where the first step is applying a [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) and the second step is either a lasso or ridge model. Because these models have a hyperparameter controlling regularization strength, you'll want to use the [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) and [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) models, which will select values for the regularization strength using cross-validation.

In [12]:
pipe = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('reg', LassoCV())
    ]
).fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print(f'MSE : {mean_squared_error(y_test, y_pred)}')
print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
print(f'R2 : {r2_score(y_test, y_pred)}')

MSE : 41770331754.19049
MSE : 125737.40908969022
R2 : 0.684334902842888


In [13]:
pipe = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('reg', RidgeCV())
    ]
).fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print(f'MSE : {mean_squared_error(y_test, y_pred)}')
print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
print(f'R2 : {r2_score(y_test, y_pred)}')

MSE : 41757913805.39382
MSE : 125799.40550564748
R2 : 0.6844287472738219


You likely didn't see much difference between the regular linear regression model and the lasso or ridge model. Let's see what happens if we add more complexity through feature interactions. We can capture more complex relationships between the predictor variables and the target variable by multiplying the predictors together before fitting the model. For example, the interaction between sqft_living and bedrooms will let the model capture if the impact of square footage depends on the number of bedrooms.



### 6. Add [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) to your pipeline after the standard scaler. Try using degree 2 features. How does this change the performance of the regular linear regression model, the lasso model, and the ridge model?

In [26]:
lr_pipe = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('reg', LinearRegression())
        
    ]
).fit(X_train, y_train)

y_pred = lr_pipe.predict(X_test)

print(f'MSE : {mean_squared_error(y_test, y_pred)}')
print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
print(f'R2 : {r2_score(y_test, y_pred)}')

MSE : 25229878727.581535
MSE : 102612.33001637411
R2 : 0.8093337595049099


In [15]:
ridge_pipe = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('reg', RidgeCV())
        
    ]
).fit(X_train, y_train)

y_pred = ridge_pipe.predict(X_test)

print(f'MSE : {mean_squared_error(y_test, y_pred)}')
print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
print(f'R2 : {r2_score(y_test, y_pred)}')

MSE : 25216536903.299854
MSE : 102587.7820475703
R2 : 0.8094345858110766


In [16]:
lasso_pipe = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(degree=2)),
        ('reg', LassoCV())
        
    ]
).fit(X_train, y_train)

y_pred = lasso_pipe.predict(X_test)

print(f'MSE : {mean_squared_error(y_test, y_pred)}')
print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
print(f'R2 : {r2_score(y_test, y_pred)}')

MSE : 25135804989.076477
MSE : 102329.79046095588
R2 : 0.810044689836512


The lasso penalty tends to cause some coeffients to zero out, so it can be viewed as a method of automatic feature selection.

### 7. Look at the set of coefficients for the lasso model. What percentage of the coefficients are zero? What are the largest non-zero coefficients?

In [17]:
lasso_pipe.named_steps['reg']

,eps,0.001
,n_alphas,'deprecated'
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.0001
,copy_X,True
,cv,None
,verbose,False
,n_jobs,None


In [18]:
lasso_pipe['reg'].coef_

array([ 0.00000000e+00, -0.00000000e+00,  1.31720760e+04,  9.50779504e+04,
        0.00000000e+00, -3.78724641e+03,  0.00000000e+00,  0.00000000e+00,
        2.50416285e+04,  9.69536317e+04,  8.57801316e+03,  0.00000000e+00,
       -1.85496916e+04,  0.00000000e+00,  7.03014342e+04, -2.39443494e+04,
        3.85747301e+04, -0.00000000e+00,  1.80582499e+02, -0.00000000e+00,
       -2.03969236e+02, -0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
       -0.00000000e+00,  5.98244604e+02, -0.00000000e+00, -0.00000000e+00,
       -6.06571436e+03, -0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00, -0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        9.66498415e+02, -0.00000000e+00, -0.00000000e+00,  5.16176706e+03,
       -0.00000000e+00, -0.00000000e+00,  1.06459315e+04,  3.20920381e+03,
        0.00000000e+00,  0.00000000e+00, -0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  2.29638792e+03, -0.00000000e+00,  0.00000000e+00,
       -6.27306164e+03,  

In [19]:
lasso_pipe[:-1].get_feature_names_out()

array(['1', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long',
       'sqft_living15', 'sqft_lot15', 'bedrooms^2', 'bedrooms bathrooms',
       'bedrooms sqft_living', 'bedrooms sqft_lot', 'bedrooms floors',
       'bedrooms waterfront', 'bedrooms view', 'bedrooms condition',
       'bedrooms grade', 'bedrooms sqft_above', 'bedrooms sqft_basement',
       'bedrooms yr_built', 'bedrooms yr_renovated', 'bedrooms lat',
       'bedrooms long', 'bedrooms sqft_living15', 'bedrooms sqft_lot15',
       'bathrooms^2', 'bathrooms sqft_living', 'bathrooms sqft_lot',
       'bathrooms floors', 'bathrooms waterfront', 'bathrooms view',
       'bathrooms condition', 'bathrooms grade', 'bathrooms sqft_above',
       'bathrooms sqft_basement', 'bathrooms yr_built',
       'bathrooms yr_renovated', 'bathrooms lat', 'bathrooms long',
       'bathrooms sqft_living

In [20]:
lasso_coefficients = pd.DataFrame({
    'variable' :lasso_pipe[:-1].get_feature_names_out(),
    'coefficient': lasso_pipe['reg'].coef_
})

In [21]:
lasso_coefficients

,variable,coefficient
0,1,0.000000
1,bedrooms,-0.000000
2,bathrooms,13172.076008
3,sqft_living,95077.950449
4,sqft_lot,0.000000
...,...,...
166,long sqft_living15,-9685.614544
167,long sqft_lot15,1026.233203
168,sqft_living15^2,4312.898385
169,sqft_living15 sqft_lot15,-683.007522


In [22]:
(lasso_coefficients['coefficient'] != 0).sum()

np.int64(88)

In [23]:
(lasso_coefficients['coefficient'] == 0).sum()

np.int64(83)

In [24]:
lasso_coefficients.sort_values('coefficient')

,variable,coefficient
161,lat^2,-41215.451933
154,yr_built sqft_living15,-25451.428122
15,long,-23944.349431
132,grade long,-21063.775277
12,yr_built,-18549.691563
...,...,...
16,sqft_living15,38574.730072
57,sqft_living grade,39077.771105
14,lat,70301.434205
3,sqft_living,95077.950449


In [27]:
lr_coefficients = pd.DataFrame({
    'variable' :lr_pipe[:-1].get_feature_names_out(),
    'coefficient': lr_pipe['reg'].coef_
})

In [29]:
lr_coefficients

,variable,coefficient
0,1,-1.242515e-11
1,bedrooms,-5.110557e+03
2,bathrooms,2.057886e+04
3,sqft_living,5.638424e+04
4,sqft_lot,1.587798e+04
...,...,...
166,long sqft_living15,-8.135574e+03
167,long sqft_lot15,5.166906e+03
168,sqft_living15^2,5.265468e+03
169,sqft_living15 sqft_lot15,9.672544e+02


In [31]:
lr_coefficients.sort_values('coefficient')

,variable,coefficient
13,yr_renovated,-1.105425e+06
161,lat^2,-4.387246e+04
154,yr_built sqft_living15,-3.480344e+04
15,long,-2.664900e+04
152,yr_built lat,-2.262151e+04
...,...,...
10,sqft_above,5.274878e+04
3,sqft_living,5.638424e+04
14,lat,7.197949e+04
9,grade,9.415629e+04


In [32]:
(lr_coefficients['coefficient'] == 0).sum()

np.int64(0)

### 8. A new hyperparameter that we have is the degree of the polynomial we're using. So that we're not overfitting to the test set, we need to use cross-validation to select this value. Set up a [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) to try out polynomial degrees from 1 to 3 and to try LinearRegression, LassoCV, and RidgeCV models. Use 'neg_mean_squared_error' as the error_score. Which combination does it find does the best?

In [43]:
pipeline = Pipeline(
    steps = [
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures()),
        ('model', LinearRegression())
        
    ]
)

param_grid = {
    "poly__degree" : [1, 2, 3],
    "model" : [LinearRegression(), LassoCV(max_iter = 10000), RidgeCV()],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_mean_squared_error", verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best model:", grid_search.best_estimator_)

best_model = grid_search.best_estimator_

# y_pred = lr_pipe.predict(X_test)

# print(f'MSE : {mean_squared_error(y_test, y_pred)}')
# print(f'MSE : {mean_absolute_error(y_test, y_pred)}')
# print(f'R2 : {r2_score(y_test, y_pred)}')

Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV] END ...........model=LinearRegression(), poly__degree=1; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=1; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=1; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=1; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=1; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=2; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=2; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=2; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=2; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=2; total time=   0.0s
[CV] END ...........model=LinearRegression(), poly__degree=3; total time=   1.1s
[CV] END ...........model=LinearRegression(), pol

In [44]:
grid_search.best_params_

{'model': LassoCV(max_iter=10000), 'poly__degree': 2}

In [45]:
y_pred = best_model.predict(X_test)
print(f'MSE:{mean_squared_error(y_test, y_pred)}')
print(f'RMSE:{root_mean_squared_error(y_test, y_pred)}')
print(f'MAE:{mean_absolute_error(y_test, y_pred)}')
print(f'MAPE:{mean_absolute_percentage_error(y_test, y_pred)}')
print(f'R2:{r2_score(y_test, y_pred)}')

MSE:25135804989.076477
RMSE:158542.75445152476
MAE:102329.79046095588
MAPE:0.21293424656174073
R2:0.810044689836512
